In [ ]:
import os
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import json
import socket
import re

from utils_mitgcm import *
from utils_eof import *

In [ ]:
def open_synthetic_ds_from_date(config, synthetic_datapath, date_str):
    datapath = f"{synthetic_datapath}_{date_str}"
    gridpath = config['gridpath']
    ref_date = config['ref_date']
    dt_mitgcm_results = config['dt']
    endian = config['endian']

    return config, open_mitgcm_ds(datapath, gridpath, ref_date, dt_mitgcm_results, endian)

In [ ]:
synthetic_datapath =  "/storage/alplakes_test/neuchatel_100m_EOF_synthetic/outputs"

In [ ]:
lake = 'neuchatel'
model = f'{lake}_2025'

In [ ]:
with open('../../../config.json', 'r') as file:
    config = json.load(file)[socket.gethostname()][model]
base_folder_path = os.path.dirname(config['datapath'])

In [ ]:
output_folder = os.path.join(base_folder_path, "eof", "synthetic_events")

# Project EOFs on continuous simulation

In [ ]:
mitgcm_config, ds_full = open_mitgcm_ds_from_config('../../../config.json', model)
ds_full, u_full, v_full, w_full = align_coordinates(ds_full)

In [ ]:
base_folder_eof_patterns = os.path.join(base_folder_path, "eof", "synthetic_events")

In [ ]:
base_folder_eof_patterns

In [ ]:
# Extract and list the dates of the extracted EOF patterns
_all_eof_subfolders = [
    name for name in os.listdir(base_folder_eof_patterns)
    if os.path.isdir(os.path.join(base_folder_eof_patterns, name))
]

_pat = re.compile(r"(\d{4}-\d{2}-\d{2})$")

eof_folders = [name for name in _all_eof_subfolders if _pat.match(name)]
eof_dates_str = [m.group(1) for name in eof_folders for m in [_pat.match(name)]]
eof_dates = pd.to_datetime(eof_dates_str)

In [ ]:
def process_eof_projection(i_eof, eof_dates_str, eof_dates, u_full, v_full, base_folder_eof_patterns, output_folder):
    eof_date_str = eof_dates_str[i_eof]
    eof_date = eof_dates[i_eof]

    ds_eof = xr.open_dataset(os.path.join(base_folder_eof_patterns, eof_date_str, f"{eof_date_str}_eof.nc"))

    u_sel = u_full.sel(time=slice(eof_date-pd.Timedelta(days=10), eof_date+pd.Timedelta(days=10)))
    v_sel = v_full.sel(time=slice(eof_date-pd.Timedelta(days=10), eof_date+pd.Timedelta(days=10)))

    pc_proj, ke_proj = project_rotary_mode(u_sel, v_sel, 100*100, ds_eof.u_eof, ds_eof.v_eof, normalize=True)

    (ke_proj/1e6).to_dataframe(name='kinetic_energy_[MJ]').reset_index().to_csv(os.path.join(output_folder, eof_date_str, f"ke_projected_eof.csv"))

    mode=0

    # amplitude
    plt.figure(figsize=(10,4))
    plt.plot(pc_proj.time, pc_proj, label='Amplitude')
    plt.ylabel('Amplitude (sqrt(KE))')
    plt.title(f'PC amplitude, eof {eof_date_str}, mode {mode+1}')
    plt.grid()
    plt.tight_layout()
    plt.savefig(os.path.join(output_folder, eof_date_str, f"projected_pc_amplitude_eof_mode{mode+1}.png"))

    # phase (rotation)
    plt.figure(figsize=(10,4))
    plt.plot(pc_proj.time, np.angle(pc_proj), label='Phase')
    plt.ylabel('Phase (radians)')
    plt.title(f'PC phase, eof {eof_date_str}, mode {mode+1}')
    plt.grid()
    plt.tight_layout()
    plt.savefig(os.path.join(output_folder, eof_date_str, f"projected_pc_phase_eof_mode{mode+1}.png"))

    plt.figure(figsize=(10,4))
    plt.plot(ke_proj.time, ke_proj/1e6, label='KE projection')
    plt.title(f'Kinetic Energy, eof {eof_date_str}, mode {mode+1}')
    plt.ylabel('KE (MJ)')
    plt.tight_layout()
    plt.savefig(os.path.join(output_folder, eof_date_str, f"projected_ke_eof_mode{mode+1}.png"))

In [ ]:
for i_eof in range(7, len(eof_dates_str)):
    print(f"Processing EOF {eof_dates_str[i_eof]} ({i_eof+1}/{len(eof_dates_str)})")
    process_eof_projection(i_eof, eof_dates_str, eof_dates, u_full, v_full, base_folder_eof_patterns, output_folder)